In [ ]:
import tensorflow as tf
import keras
import numpy as np
import glob
import matplotlib.pyplot as plt
import importlib
from absl import logging

# Define o nível de logging para silenciar os avisos do TF
tf.get_logger().setLevel('ERROR')
logging.set_verbosity(logging.ERROR)

# Vetorização de Texto: de regras manuais ao Keras TextVectorization

Redes neurais só enxergam números. Por isso, antes de qualquer modelo, é preciso transformar texto em sequências numéricas.

1. vetorizador manual (do zero)
2. `keras.layers.TextVectorization`
3. N-gram
4. TF-IDF.

## 1. Vetorizador manual (do zero)

Para entender o que uma camada de vetorização faz por baixo dos panos, vamos implementar uma versão dummy com python puro. 

O vetorizador de texto pode ter 4 responsabilidades:

- **standardize**: padronizar caracteres e estrutura sintática (ex.: minúsculas, sem pontuação);
- **tokenize**: quebrar a sentença em tokens (aqui, palavras separadas por espaço);
- **vocabulary**: construir um vocabulário `token -> índice` a partir de um corpus;
- **encode / decode**: converter sentenças em sequências de índices, e voltar.

In [ ]:
import string


class Vectorizer:
    """Vetorizador de texto dummy: padroniza, tokeniza e mapeia palavras para índices inteiros."""

    def __init__(self):
        # índice 0: reservado para padding (usado para completar sentenças curtas até um tamanho fixo)
        # índice 1: reservado para [UNK], qualquer palavra fora do vocabulário
        self.vocabulary = {"": 0, "[UNK]": 1}

    def standardize(self, sentence):
        """Minúsculas + remoção de pontuação"""
        sentence = sentence.lower()
        return "".join(char for char in sentence if char not in string.punctuation)

    def tokenize(self, sentence):
        """Sentença padronizada -> lista de palavras (split por espaço)."""
        sentence = self.standardize(sentence)
        return sentence.split()

    def make_vocabulary(self, data):
        """Expande o vocabulário atual com os tokens de um corpus (lista de sentenças)."""
        for sentence in data:
            tokens = self.tokenize(sentence)
            for token in tokens:
                # cada palavra nova recebe o próximo índice livre da sequência
                if token not in self.vocabulary:
                    self.vocabulary[token] = len(self.vocabulary)

        # vocabulário reverso (índice -> palavra), necessário para decodificar
        self.inverse_vocabulary = dict(enumerate(self.vocabulary.keys()))

    def encode(self, sentence):
        """Sentença -> lista de índices. Palavras fora do vocabulário viram [UNK] (índice 1)."""
        tokens = self.tokenize(sentence)
        return [self.vocabulary.get(token, 1) for token in tokens]

    def decode(self, int_sequence):
        """Lista de índices -> sentença reconstruída (aproximada: pontuação e capitalização já foram perdidas na padronização)."""
        return " ".join(self.inverse_vocabulary.get(i, "[UNK]") for i in int_sequence)

### 1.1 Vocabulário inicial

Ao instanciar a classe, o vocabulário contém apenas os dois tokens especiais definidos no `__init__`:

In [ ]:
vectorizer = Vectorizer()

vectorizer.vocabulary

### 1.2 Construindo o vocabulário a partir de um corpus

Um pequeno corpus com repetições propositais. Repare que sentenças repetidas não criam novas entradas; o vocabulário só cresce com palavras **inéditas**:

In [ ]:
data = [
    "No meio do caminho tinha uma pedra",
    "tinha uma pedra no meio do caminho",
    "tinha uma pedra",
    "no meio do caminho tinha uma pedra.",
    "Nunca me esquecerei desse acontecimento",
    "na vida de minhas retinas tão fatigadas.",
    "Nunca me esquecerei que no meio do caminho",
    "tinha uma pedra",
    "tinha uma pedra no meio do caminho",
    "no meio do caminho tinha uma pedra.",
    "- Carlos Drummond de Andrade."
]
vectorizer.make_vocabulary(data)
vectorizer.vocabulary

`inverse_vocabulary` é o dicionário `índice -> token`, construído dentro de `make_vocabulary` e usado pelo `decode`:

In [ ]:
vectorizer.inverse_vocabulary

### 1.3 Encode / decode

Testando com uma sentença **fora do corpus de treino**, com pequenas variações: "havia" no lugar de "tinha", e "pedras" vez de "pedra". Como essas palavras nunca apareceram no `make_vocabulary`, viram `[UNK]` (índice 1):

> se tivéssemos configurado um stemming de pedras -> pedra, seria possível não perder essa ocorrência, por exemplo. às vezes, menos informação tem maior valor prático.

In [ ]:
sentence = "No meio do caminho havia uma pedras."

encoded_sentence = vectorizer.encode(sentence)
encoded_sentence

In [ ]:
decoded_sentence = vectorizer.decode(encoded_sentence)
decoded_sentence

O `decode` não recupera a sentença original: pontuação e capitalização já haviam sido descartadas em `standardize`, e as palavras desconhecidas viraram `[UNK]`. A vetorização é **uma transformação com perdas**.

## 2. Keras TextVectorization

A implementação manual funciona, mas roda em Python puro, palavra por palavra. `keras.layers.TextVectorization` faz o mesmo trabalho | standardize → tokenize → vocabulário → encode | só que:

- paralelizada, o que importa muito em corpora grandes;
- integrável direto em um `tf.data.Dataset` ou como a primeira camada de um `keras.Model`.

NOTE: por depender de operações de dicionário (hash tables), essa camada roda em **CPU**, não GPU.

In [ ]:
# camada de vetorização com configuração default
vectorizer = keras.layers.TextVectorization()

Assim como o vetorizador manual, `TextVectorization` aceita funções de `standardize` e `split` (tokenização) customizadas. Abaixo, replicamos o mesmo comportamento de antes (minúsculas + remoção de pontuação) usando operações do TensorFlow, necessárias para rodar dentro do grafo da camada:

In [ ]:
import re
import string


def custom_standardization_fn(string_tensor):
    """Minúsculas + remoção de pontuação, com operações tf.strings (equivalente ao standardize manual)."""
    lowercase_string = tf.strings.lower(string_tensor)
    return tf.strings.regex_replace(
        lowercase_string, f"[{re.escape(string.punctuation)}]", ""
    )


def custom_split_fn(string_tensor):
    """Split por espaço, com tf.strings (equivalente ao tokenize manual)."""
    return tf.strings.split(string_tensor)


vectorizer = keras.layers.TextVectorization(
    output_mode="int",
    standardize=custom_standardization_fn,
    split=custom_split_fn,
)

### 2.1 Construindo o vocabulário: `adapt()`

Diferente do `make_vocabulary` manual, aqui chamamos `adapt()`, o mesmo método usado por camadas de normalização do Keras. Antes de adaptar, o vocabulário contém só os tokens especiais:

In [ ]:
# vocabulário inicial, antes do adapt()
vectorizer.get_vocabulary()

In [ ]:
# adapt() varre o corpus e constrói o vocabulário (equivalente ao make_vocabulary manual)
vectorizer.adapt(data)
vectorizer.get_vocabulary()

### 2.2 Encode

A camada é chamada diretamente, como uma função: `vectorizer(sentence)` já retorna a sequência de índices.

In [ ]:
sentence = "No meio do caminho havia uma pedra."
encoded_sentence = vectorizer(sentence)
encoded_sentence

In [ ]:
# keras.layers.TextVectorization não guarda um vocabulário reverso (não foi feita para decodificar)
# se precisarmos decodificar, construímos esse mapeamento manualmente, assim como antes
vocab = vectorizer.get_vocabulary()
invocab = dict(enumerate(vocab))

In [ ]:
vocab

In [ ]:
invocab

In [ ]:
# .numpy() extrai o valor do tensor antes de consultar o dicionário
decoded_sentence = " ".join(invocab.get(i.numpy(), "[UNK]") for i in encoded_sentence)
decoded_sentence

## 3. N-gram

Até aqui, a ordem das palavras não importa para o vocabulário: "tinha uma pedra" e "pedra uma tinha" teriam a mesma representação bag-of-words. **N-grams** recuperam um pouco dessa ordem quando N > 1: além de palavras isoladas (1-gramas), também indexamos pares consecutivos (bigramas, N=2), trios (trigramas, N=3), etc.

`output_mode='multi_hot'` produz um vetor binário do tamanho do vocabulário: `1` se o token (ou n-grama) aparece na sentença, `0` caso contrário. É a representação bag-of-words (aqui, bag-of-n-grams).

In [ ]:
vectorizer = keras.layers.TextVectorization(
    ngrams=2,
    output_mode='multi_hot',
)
vectorizer.adapt(data)

sentence = 'no meio do caminho havia'
encoded = vectorizer(sentence)
encoded

In [ ]:
# vocabulário maior que antes: agora inclui palavras isoladas E pares consecutivos (bigramas)
len(vectorizer.get_vocabulary())

In [ ]:
dict(enumerate(vectorizer.get_vocabulary()))

## 4. TF-IDF

**T**erm **F**requency, **I**nverse **D**ocument **F**requency: pondera cada token pela sua frequência na sentença (TF), dividida pela frequência com que aparece no corpus inteiro (IDF).

Tokens muito comuns em todo o corpus (ex.: "do", "no") recebem peso baixo; tokens raros e mais distintivos recebem peso alto.

> OBS: na prática, o keras utiliza `tf * ln(idf)` para evitar a inflação de termos muito frequentes

Relembrando o corpus de treino:

In [ ]:
data

In [ ]:
vectorizer = keras.layers.TextVectorization(
    ngrams=3,
    output_mode='tf-idf',
)
vectorizer.adapt(data)

sentence = 'no meio do caminho tinha uma pedra'
vectorizer(sentence)

In [ ]:
vectorizer.get_vocabulary()